In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
from huggingface_hub import snapshot_download

snapshot_download(repo_id="Lacito/pangloss", 
                  repo_type="dataset", local_dir="./pangloss")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 15 files: 100%|██████████| 15/15 [00:04<00:00,  3.11it/s]


'/home/ubuntu/pangloss'

In [3]:
files = glob('pangloss/*/*.parquet')
len(files)

13

In [5]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in tqdm(files):
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in range(len(df)):
            t = df['sentence'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}_{df['speaker'].iloc[i]}"
            })
        
    return data

In [6]:
data = multiprocessing(files, loop, len(files))

100%|██████████| 1/1 [01:46<00:00, 106.89s/it]


In [7]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'pangloss_audio/pangloss-japh1234-train-00006-of-00007_0.mp3',
 'text': 'pɯ-kɯ-nɯʑɯβ to-ʑɣɤpa.',
 'speaker': 'pangloss_audio_Tshendzin'}

In [8]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'pangloss')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 65.17ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████| 1.94MB / 1.94MB,  215kB/s  
Processing Files (1 / 1): 100%|██████████| 1.94MB / 1.94MB,  211kB/s  
New Data Upload: 100%|██████████| 1.94MB / 1.94MB,  211kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:09<00:00,  9.55s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/41828dd905f4ef8f92a311773bc67298b7ca2d83', commit_message='Upload dataset', commit_description='', oid='41828dd905f4ef8f92a311773bc67298b7ca2d83', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [9]:
audio_files = [d['audio_filename'] for d in data]

with open('pangloss-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [12]:
# !zip -rq pangloss_audio.zip pangloss_audio

In [13]:
# !hf upload malaysia-ai/Multilingual-TTS pangloss_audio.zip --repo-type=dataset

In [16]:
# !zip -rq pangloss_audio_neucodec.zip pangloss_audio_neucodec

In [17]:
# !hf upload malaysia-ai/Multilingual-TTS pangloss_audio_neucodec.zip --repo-type=dataset